# Neural networks — Lab 1: from a numpy perceptron to a convolutional network

In this lab you build a two-layer neural network by hand in numpy, train it to tell birds from dogs, and then let Keras do the same job with a convolutional network. Along the way you meet the ideas that Lecture 2 builds on: the forward and backward pass, the loss curve, overfitting, dropout and early stopping. At the end you apply what you learned to the MNIST digits.

**Outline**

- Part A. A two-layer network in numpy (bird vs dog, 32×32 images flattened to 3072 numbers)
- Part B. The same problem with a convolutional network in Keras, with and without dropout
- Part C. A small data set: 19 face images, happy or not happy, and transfer learning
- Part D. MNIST handwritten digits (task)

Cells marked **Task** contain `# your code here`. Everything else runs as given.

## Setup

On Colab, choose *Runtime → Change runtime type → GPU* before you start; the Keras parts run faster.

The cell below imports Keras. Keras 3 can run on TensorFlow (Colab, most local installs) or on PyTorch; if TensorFlow is missing we switch to the PyTorch backend, the rest of the notebook is identical. It then finds the training images: on Colab it clones the course repository, otherwise it looks for a `train` folder next to the notebook.

In [ ]:
import os, sys
try:
    import tensorflow  # present on Colab and most local installs
except ImportError:
    os.environ["KERAS_BACKEND"] = "torch"   # Keras 3 also runs on PyTorch
import keras
from keras import layers
print("Keras", keras.__version__, "backend:", keras.backend.backend())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(0)   # numpy, python and backend seeds, so results repeat

# --- locate the training images -------------------------------------------------------
if not Path("Bern02/Labs/Neural_Networks/train").exists() and "google.colab" in sys.modules:
    !git clone https://github.com/luchem/Bern02.git --depth=1
candidates = ["Bern02/Labs/Neural_Networks/train", "train",
              "/home/user/workspace/nn_data/bern02/Labs/Neural_Networks/train"]
DATA = next(Path(p) for p in candidates if Path(p).exists())
print("data folder:", DATA.resolve())

## Part A. A two-layer network in numpy

An artificial neuron has six parts, from input to output:

1. Input values $x_1, \dots, x_D$ (here: the pixel values of an image).
2. One weight $w_i$ per input.
3. A weighted sum $z = \sum_{i=1}^{D} w_i x_i + b$. The constant $b$ is the *bias*; it shifts the threshold.
4. An *activation function* $g(z)$ that turns the sum into the neuron's output. A hard step ($0$ if $z \le 0$, $1$ if $z > 0$) is not differentiable, so one uses smooth or piecewise-linear functions instead: the logistic (sigmoid) function $\sigma(z) = 1/(1+e^{-z})$, or the rectified linear unit $\mathrm{ReLU}(z) = \max(0, z)$.
5. The output $a = g(z)$.
6. Connections to the next layer, where $a$ becomes an input.

A *layer* is a set of such neurons that share the same inputs; a network stacks layers. The plot shows the two activation functions we use.

In [ ]:
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(z, 1/(1 + np.exp(-z)));  ax[0].set_title(r'sigmoid  $\sigma(z)=1/(1+e^{-z})$')
ax[1].plot(z, np.maximum(0, z));     ax[1].set_title(r'ReLU  $\max(0,z)$')
for a in ax:
    a.set_xlabel('z'); a.set_ylabel('g(z)')
plt.tight_layout()

### Loading the data

The folder `train` holds 1200 bird and 1200 dog images of 64×64 pixels. We shrink them to 32×32 pixels (three colour channels), scale the pixel values to $[0, 1]$, and use label 0 for bird and 1 for dog.

In [ ]:
IMAGE_SIZE = 32

def load_folder(folder, size=IMAGE_SIZE):
    "Load every jpg in `folder` as an RGB array of shape (size, size, 3) scaled to [0, 1]."
    files = sorted(Path(folder).glob("*.jpg"))
    return np.array([np.asarray(Image.open(p).convert("RGB").resize((size, size)))
                     for p in files], dtype=np.float32) / 255.0

birds = load_folder(DATA / "bird")
dogs  = load_folder(DATA / "dog")
X = np.concatenate([birds, dogs])                    # (N, 32, 32, 3)
y = np.array([0]*len(birds) + [1]*len(dogs))        # 0 = bird, 1 = dog
print("images:", X.shape, " labels:", y.shape, " birds:", len(birds), " dogs:", len(dogs))

### Looking at the data

How many images do we have in total, and what is the shape of one image? Look at a few: at 32×32 pixels the pictures are coarse, but you can still tell a bird from a dog. Whether the network can is the question of this lab.

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(10, 3.6))
for i, a in enumerate(ax[0]):
    a.imshow(birds[i]); a.set_title("bird"); a.axis("off")
for i, a in enumerate(ax[1]):
    a.imshow(dogs[i]); a.set_title("dog"); a.axis("off")
plt.tight_layout()
print("one image has shape", X[0].shape, "=", X[0].size, "numbers")

### Train/test split and flattening

We hold out 20 % of the images as a *test set* that the network never sees during training. Only the test accuracy tells us whether the network learned something general or just memorised the training pictures. `train_test_split` with `stratify=y` keeps the bird/dog ratio equal in both parts.

The two-layer network connects every pixel to every hidden neuron, so the position of a pixel carries no special meaning. We therefore *flatten* each image to a vector of $32 \cdot 32 \cdot 3 = 3072$ numbers. Following the usual convention for hand-written networks, the data matrix has one **column** per image: shape `(3072, m)`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

Xf_train = X_train.reshape(len(X_train), -1).T.astype(np.float64)   # (3072, m_train)
Xf_test  = X_test.reshape(len(X_test), -1).T.astype(np.float64)     # (3072, m_test)
Y_train  = y_train.reshape(1, -1)                  # (1, m_train)
Y_test   = y_test.reshape(1, -1)
print("train:", Xf_train.shape, Y_train.shape, "  test:", Xf_test.shape, Y_test.shape)

### The network and its equations

We build a network with one hidden layer of $n_h$ ReLU neurons and one sigmoid output neuron. With the data matrix $X$ of shape $(n_x, m)$ and labels $Y$ of shape $(1, m)$, the **forward pass** is

$$Z_1 = W_1 X + b_1, \qquad A_1 = \mathrm{ReLU}(Z_1), \qquad Z_2 = W_2 A_1 + b_2, \qquad A_2 = \sigma(Z_2).$$

$A_2$ is the predicted probability that each image is a dog. The **loss** is the binary cross-entropy, averaged over the $m$ images,

$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log a_i + (1 - y_i) \log (1 - a_i) \right].$$

The **backward pass** applies the chain rule from the output back to the input. For the sigmoid output with cross-entropy loss the first step is remarkably simple,

$$dZ_2 = A_2 - Y, \qquad dW_2 = \tfrac{1}{m}\, dZ_2 A_1^{T}, \qquad db_2 = \tfrac{1}{m} \sum_i dZ_2,$$

and for the hidden layer

$$dA_1 = W_2^{T} dZ_2, \qquad dZ_1 = dA_1 \odot \mathbb{1}[Z_1 > 0], \qquad dW_1 = \tfrac{1}{m}\, dZ_1 X^{T}, \qquad db_1 = \tfrac{1}{m} \sum_i dZ_1.$$

Here $dW$ is short for $\partial J / \partial W$ and $\odot$ is element-wise multiplication. Gradient descent then updates every parameter as $W \leftarrow W - \eta\, dW$ with learning rate $\eta$. The code below implements exactly these lines.

In [ ]:
def initialize_parameters(n_x, n_h, n_y):
    "Random small weights, zero biases. Shapes: W1 (n_h, n_x), b1 (n_h, 1), W2 (n_y, n_h), b2 (n_y, 1)."
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

In [ ]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def forward(X, p):
    "Forward pass. Returns the output A2 and a cache of intermediate values for the backward pass."
    Z1 = p["W1"] @ X + p["b1"]
    A1 = relu(Z1)
    Z2 = p["W2"] @ A1 + p["b2"]
    A2 = sigmoid(Z2)
    return A2, (X, Z1, A1, A2)

In [ ]:
def compute_cost(A2, Y):
    "Binary cross-entropy averaged over the m examples."
    m = Y.shape[1]
    A2 = np.clip(A2, 1e-12, 1 - 1e-12)          # avoid log(0)
    return float(-np.sum(Y*np.log(A2) + (1 - Y)*np.log(1 - A2)) / m)

Training is a loop of three steps:

1. **Forward pass**: the data flows through the network and produces the outputs $A_2$.
2. **Loss**: compare $A_2$ with the true labels.
3. **Backward pass**: compute the gradient of the loss with respect to every weight and bias, then move each parameter a small step against its gradient.

Backpropagation is nothing more than the chain rule organised so that each layer reuses what the layer after it already computed.

In [ ]:
def backward(p, cache, Y):
    "Backward pass: gradients of the cost with respect to W1, b1, W2, b2."
    X, Z1, A1, A2 = cache
    m = X.shape[1]
    dZ2 = A2 - Y                                   # sigmoid + cross-entropy
    dW2 = dZ2 @ A1.T / m
    db2 = dZ2.sum(axis=1, keepdims=True) / m
    dA1 = p["W2"].T @ dZ2
    dZ1 = dA1 * (Z1 > 0)                           # derivative of ReLU
    dW1 = dZ1 @ X.T / m
    db1 = dZ1.sum(axis=1, keepdims=True) / m
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

In [ ]:
def update_parameters(p, grads, learning_rate):
    "One gradient-descent step on every parameter."
    for name in ("W1", "b1", "W2", "b2"):
        p[name] = p[name] - learning_rate * grads["d" + name]
    return p

The output $A_2$ is a probability. To turn it into a class we compare it with 0.5: `(A2 > 0.5)` gives `True` for dog and `False` for bird, and `astype(int)` makes that 1 and 0. (Older versions of this notebook used `np.floor(A2 + 0.5)`, which does the same but hides the threshold.)

In [ ]:
def predict(X, p):
    "Class labels (0 = bird, 1 = dog) for the images in X."
    A2, _ = forward(X, p)
    return (A2 > 0.5).astype(int)

def accuracy(X, Y, p):
    return float(np.mean(predict(X, p) == Y))

### Training loop

`two_layer_model` puts the pieces together. Two details:

- The learning rate is multiplied by 0.999 after every iteration. This is a *learning-rate schedule*: large steps early, smaller steps as we approach a minimum. Lecture 2 explains why this helps.
- Every `record_every` iterations we store the cost and the train and test accuracy so that we can plot them afterwards.

In [ ]:
def two_layer_model(X, Y, n_h=128, learning_rate=0.05, num_iterations=1500, record_every=50,
                    X_test=None, Y_test=None, verbose=True):
    "Train LINEAR -> RELU -> LINEAR -> SIGMOID by full-batch gradient descent."
    n_x, n_y = X.shape[0], 1
    p = initialize_parameters(n_x, n_h, n_y)
    history = {"iteration": [], "cost": [], "train_acc": [], "test_acc": []}
    for i in range(num_iterations + 1):
        A2, cache = forward(X, p)
        if i % record_every == 0 or i == num_iterations:
            history["iteration"].append(i)
            history["cost"].append(compute_cost(A2, Y))
            history["train_acc"].append(accuracy(X, Y, p))
            if X_test is not None:
                history["test_acc"].append(accuracy(X_test, Y_test, p))
            if verbose and i % (5*record_every) == 0:
                print(f"iteration {i:5d}  cost {history['cost'][-1]:.4f}  train acc {history['train_acc'][-1]:.3f}"
                      + (f"  test acc {history['test_acc'][-1]:.3f}" if X_test is not None else ""))
        if i == num_iterations:
            break
        grads = backward(p, cache, Y)
        learning_rate *= 0.999                       # learning-rate schedule (decay per iteration)
        p = update_parameters(p, grads, learning_rate)
    return p, pd.DataFrame(history)

In [ ]:
np.random.seed(0)
n_x = IMAGE_SIZE * IMAGE_SIZE * 3        # 3072 inputs
params, hist = two_layer_model(Xf_train, Y_train, n_h=128, learning_rate=0.05, num_iterations=1500,
                               X_test=Xf_test, Y_test=Y_test)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist.iteration, hist.cost)
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("cross-entropy loss (train)")
ax[1].plot(hist.iteration, hist.train_acc, "o-", ms=3, label="train")
ax[1].plot(hist.iteration, hist.test_acc, "o-", ms=3, label="test")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("accuracy"); ax[1].legend()
plt.tight_layout()
print(f"final train accuracy {hist.train_acc.iloc[-1]:.3f}   test accuracy {hist.test_acc.iloc[-1]:.3f}")

Read the two panels together. The training loss falls steadily and the training accuracy keeps rising, but the test accuracy levels off (or drops) well before the end. From that point on the network memorises the training images instead of learning what makes a dog a dog: it **overfits**. Chance level is 50 %, so the network did learn something, but a dense layer with 3072 inputs is a blunt tool for images. Part B shows a better one.

**Task A1.** Change the size of the hidden layer (for example 10, 50, 200 units) and the learning rate (for example 0.01, 0.05, 0.2). For each setting report the final train and test accuracy in a small table (a `pandas.DataFrame` is convenient). Use fewer iterations (say 600) to keep the run short. Which setting generalises best? Does a larger hidden layer help?

In [ ]:
# Task A1: loop over a few (n_h, learning_rate) pairs, train with two_layer_model(..., verbose=False)
# and collect the final train/test accuracies in a table.
# your code here

**Task A2.** Use `predict` on the test set, find the images that the network gets wrong, and show the first 12 of them in a grid with the predicted and the true class in the title. What do the misclassified pictures have in common?

In [ ]:
# Task A2: find misclassified test images and show up to 12 of them (predicted / true label in the title).
names = {0: "bird", 1: "dog"}
# your code here

## Part B. A convolutional network in Keras

A dense layer treats the 3072 pixel values as unrelated numbers. A **convolutional network** (CNN) instead slides a small filter (here 3×3 pixels) over the image and computes, at every position, the weighted sum of the pixels under the filter. One filter therefore has only $3 \cdot 3 \cdot 3 + 1 = 28$ parameters, yet it looks at the whole image. Each filter produces a *feature map*; a layer with 32 filters produces 32 maps. The building blocks are:

1. `layers.Conv2D(32, (3, 3), padding='same')`: 32 filters of 3×3; `padding='same'` keeps the image size.
2. An activation, here the *leaky ReLU*: like ReLU, but negative inputs are scaled by 0.1 instead of set to zero, so the gradient never vanishes entirely (plot below).
3. `layers.MaxPooling2D((2, 2))`: keeps the largest value in each 2×2 block, halving width and height. This reduces the number of parameters downstream and makes the network less sensitive to small shifts.
4. After a few such blocks, `layers.Flatten()` turns the maps into a vector, and dense layers make the decision. The last layer is a single sigmoid neuron, exactly as in Part A.
5. Keras trains the network with an optimizer (`Adam`, an improved gradient descent; Lecture 2) over several **epochs**. One epoch is one pass over the training set, in mini-batches of `batch_size` images; the weights are updated after every mini-batch.

In [ ]:
z = np.linspace(-2, 2, 200)
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(z, np.where(z > 0, z, 0.1*z))
ax.axvline(0, color="k", alpha=0.3)
ax.set_xlabel("z"); ax.set_ylabel("LeakyReLU(z)"); ax.set_title("leaky ReLU, negative slope 0.1")
plt.tight_layout()

### Data for Keras

Keras expects the images as an array of shape `(m, 32, 32, 3)`, which is how we loaded them. We reuse the split from Part A. Note the order of the splits: the **test set** was set aside first and is never used during training. Keras will additionally take 20 % of the *training* set as a **validation set** (`validation_split=0.2`); it monitors the loss on that part after every epoch. The validation loss guides decisions during training (when to stop), the test set gives the final, unbiased number.

In [ ]:
print("train:", X_train.shape, y_train.shape, "  test:", X_test.shape, y_test.shape)
print("class balance in train: birds", np.sum(y_train == 0), " dogs", np.sum(y_train == 1))

### Two models: without and with dropout

`build_cnn` returns the network. With `dropout=True` it inserts `Dropout` layers, which during training set a random fraction of the neurons to zero at every batch. The network can then not rely on any single neuron and has to spread the evidence; at test time all neurons are active. Dropout is a *regularisation* method (Lecture 2). The layer names matter later in Part C.

In [ ]:
def build_cnn(dropout=False, name="cnn"):
    m = keras.Sequential(name=name)
    m.add(layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
    for k, filters in enumerate([32, 64, 128]):
        m.add(layers.Conv2D(filters, (3, 3), padding="same"))
        m.add(layers.LeakyReLU(negative_slope=0.1))
        m.add(layers.MaxPooling2D((2, 2)))
        if dropout:
            m.add(layers.Dropout(0.25 if k < 2 else 0.5))
    m.add(layers.Flatten(name="features"))
    m.add(layers.Dense(128))
    m.add(layers.LeakyReLU(negative_slope=0.1))
    if dropout:
        m.add(layers.Dropout(0.3))
    m.add(layers.Dense(1, activation="sigmoid"))
    m.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
    return m

build_cnn().summary()

Look at the parameter count. The three convolutional layers together hold fewer than 100 000 weights and process the whole image; the dense layer after `Flatten` (2048 → 128) holds most of the rest. Compare this with Part A, where the first layer alone had $3072 \times 128 \approx 393\,000$ weights and saw nothing of the image structure.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

histories, models, test_acc = {}, {}, {}
for label, use_dropout in [("no dropout", False), ("dropout", True)]:
    keras.utils.set_random_seed(0)
    model = build_cnn(dropout=use_dropout, name=label.replace(" ", "_"))
    h = model.fit(X_train, y_train, batch_size=50, epochs=50, validation_split=0.2,
                  callbacks=[early_stop], verbose=0)
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    histories[label], models[label], test_acc[label] = h.history, model, acc
    print(f"{label:11s}: stopped after {len(h.history['loss']):2d} epochs, "
          f"best val_loss {min(h.history['val_loss']):.3f}, test accuracy {acc:.3f}")
    model.save(f"bird_dog_{label.replace(' ', '_')}.keras")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for label, c in [("no dropout", "C0"), ("dropout", "C1")]:
    h = histories[label]; ep = np.arange(1, len(h["loss"]) + 1)
    ax[0].plot(ep, h["loss"], "--", color=c, label=f"{label}: train")
    ax[0].plot(ep, h["val_loss"], "-", color=c, label=f"{label}: validation")
    ax[1].plot(ep, h["accuracy"], "--", color=c, label=f"{label}: train")
    ax[1].plot(ep, h["val_accuracy"], "-", color=c, label=f"{label}: validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("binary cross-entropy"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend(fontsize=8)
plt.tight_layout()

**Reading the curves.** Without dropout the training loss drops towards zero within a few epochs while the validation loss reaches a minimum and then climbs again; the gap between the dashed and solid lines is the signature of overfitting. The `EarlyStopping` callback watches `val_loss`, stops training when it has not improved for 8 epochs, and restores the weights from the best epoch. With dropout the training loss falls more slowly (part of the network is switched off at every step) and the validation loss follows it much longer, so the two curves stay close and the model trains for more epochs. The test accuracies printed above show which model generalises better; with only 1920 training images the difference is modest, and it varies from run to run.

**Task B1.** Modify `build_cnn` (copy it into the cell below under a new name) in one of two ways: add `layers.BatchNormalization()` after each `Conv2D`, or change the dropout rates (for example all 0.1, or all 0.5). Train with the same early-stopping callback, print the number of epochs and the test accuracy, and compare with the two models above. One sentence: what did the change do to the training and validation curves?

In [ ]:
# Task B1: define a variant of build_cnn (BatchNormalization or other dropout rates), train it with
# early stopping (validation_split=0.2, batch_size=50, epochs=50, verbose=0) and print epochs + test accuracy.
# your code here

**Task B2.** Take the dropout model from `models["dropout"]`, predict the probabilities for the test set (`model.predict(X_test, verbose=0)`), and show 12 misclassified images with the predicted probability of "dog" in the title. Are the mistakes confident (probability near 0 or 1) or borderline (near 0.5)?

In [ ]:
# Task B2: predicted probabilities on the test set, grid of 12 misclassified images with p(dog) in the title.
# your code here

## Part C. A small data set: 19 faces

The folder `faces` contains 19 photos of your teacher, some smiling, some not. Old versions of this lab trained a CNN from scratch on them. With 19 images that cannot work in a meaningful way: a CNN has thousands of parameters and will memorise 19 pictures perfectly while learning nothing you can trust. The honest procedure is

1. use **cross-validation**, so that every prediction is made for an image the model did not see,
2. report the accuracy **with its uncertainty**, and
3. bring in knowledge from a larger data set (**transfer learning**, Task C1).

We crop the face region as in the original notebook, average the colour channels to grey, and resize to 32×32. The labels (1 = happy) are given in the order of the sorted file names.

In [ ]:
face_files = sorted((DATA / "faces").glob("*.jpg"))
happy = np.array([0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0])
assert len(face_files) == len(happy) == 19

def load_face(p, size=IMAGE_SIZE):
    img = np.asarray(Image.open(p).convert("RGB"))[200:600, 500:800, :]     # crop to the face
    return np.asarray(Image.fromarray(img).resize((size, size)), dtype=np.float32) / 255.0

faces_rgb  = np.array([load_face(p) for p in face_files])       # (19, 32, 32, 3), for Task C1
faces_gray = faces_rgb.mean(axis=3)                             # (19, 32, 32)
print("faces:", faces_gray.shape, " happy:", happy.sum(), " not happy:", (1 - happy).sum())

fig, ax = plt.subplots(2, 10, figsize=(12, 3))
for a, img, lab, p in zip(ax.ravel(), faces_gray, happy, face_files):
    a.imshow(img, cmap="gray"); a.axis("off"); a.set_title(("happy" if lab else "not") + f" ({p.stem[-2:]})", fontsize=8)
ax.ravel()[-1].axis("off")
plt.tight_layout()

### Cross-validation and uncertainty

The majority class ("not happy", 12 of 19) already gives 63 % accuracy, so that is the number to beat. We use 5-fold stratified cross-validation: the model is trained on four fifths of the images and tested on the remaining fifth, five times, so every image is predicted once while held out.

For a classifier with true accuracy $p$ tested on $n$ images, the standard error of the measured accuracy is $\sqrt{p(1-p)/n}$. With $n = 19$ that is about 0.1: an accuracy of 0.8 means "somewhere between 0.6 and 1.0". Keep that in mind when you read the numbers.

We compare two models: logistic regression on the 1024 grey pixel values, and a tiny CNN.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

def cv_accuracy(fit_predict, X, y, n_splits=5, seed=0):
    "Cross-validated accuracy and its binomial standard error. fit_predict(X_tr, y_tr, X_te) -> labels."
    pred = np.zeros_like(y)
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(X, y):
        pred[te] = fit_predict(X[tr], y[tr], X[te])
    acc = np.mean(pred == y)
    return acc, np.sqrt(acc*(1 - acc)/len(y)), pred

def logreg_fit_predict(Xtr, ytr, Xte):
    clf = LogisticRegression(C=0.1, max_iter=2000).fit(Xtr.reshape(len(Xtr), -1), ytr)
    return clf.predict(Xte.reshape(len(Xte), -1))

def tiny_cnn_fit_predict(Xtr, ytr, Xte):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1)),
                          layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
                          layers.MaxPooling2D((4, 4)),
                          layers.Flatten(),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(1e-3))
    m.fit(Xtr[..., None], ytr, epochs=60, batch_size=8, verbose=0)
    return (m.predict(Xte[..., None], verbose=0).ravel() > 0.5).astype(int)

acc_lr,  se_lr,  _ = cv_accuracy(logreg_fit_predict,   faces_gray, happy)
acc_cnn, se_cnn, _ = cv_accuracy(tiny_cnn_fit_predict, faces_gray, happy)
print(f"majority class          : {max(happy.mean(), 1 - happy.mean()):.2f}")
print(f"logistic regression (CV): {acc_lr:.2f} ± {se_lr:.2f}")
print(f"tiny CNN            (CV): {acc_cnn:.2f} ± {se_cnn:.2f}")

**Task C1 (transfer learning).** The bird/dog CNN has learned filters that respond to edges, blobs and textures. Those are useful for any image. Build a feature extractor from the dropout model, `keras.Model(inputs=cnn.input, outputs=cnn.get_layer("features").output)`, run the 19 RGB face crops (`faces_rgb`) through it to get a 2048-number feature vector per image, and fit a `LogisticRegression` on these features with the same 5-fold `cv_accuracy`. Compare with the two results above. Is the improvement larger than the uncertainty?

In [ ]:
# Task C1: feature extractor from the bird/dog CNN -> features for faces_rgb -> logistic regression with cv_accuracy.
# your code here

### Where does the model look? (demo)

An *occlusion map* asks a simple question: if we cover a small patch of the image with grey, how much does the predicted probability of "happy" change? Patches whose covering matters most are where the model looks. This needs only `predict`, so it works on any backend. We use the transfer-learning pipeline fitted on all 19 images. (A proper saliency map uses the gradient of the output with respect to the pixels; that is a Lecture 2 topic.)

In [ ]:
clf_all = LogisticRegression(C=0.1, max_iter=5000).fit(face_features, happy)
i, patch, stride = 1, 6, 2                                  # image 02 is a smiling one
img = faces_rgb[i]
positions = [(r, c) for r in range(0, IMAGE_SIZE - patch + 1, stride)
                    for c in range(0, IMAGE_SIZE - patch + 1, stride)]
occluded = np.repeat(img[None], len(positions), axis=0)
for k, (r, c) in enumerate(positions):
    occluded[k, r:r+patch, c:c+patch, :] = 0.5
p_full = clf_all.predict_proba(extractor.predict(img[None], verbose=0))[0, 1]
p_occ  = clf_all.predict_proba(extractor.predict(occluded, verbose=0))[:, 1]
n = int(np.sqrt(len(positions)))
heat = (p_full - p_occ).reshape(n, n)

fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
ax[0].imshow(img); ax[0].set_title(f"p(happy) = {p_full:.2f}"); ax[0].axis("off")
im = ax[1].imshow(heat, cmap="coolwarm", vmin=-np.abs(heat).max(), vmax=np.abs(heat).max())
ax[1].set_title("drop in p(happy) when patch covered"); ax[1].axis("off")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout()

## Part D. MNIST handwritten digits

MNIST is the classic benchmark: 70 000 grey images of 28×28 pixels showing the digits 0–9. Keras downloads it (11 MB). To keep the run short we train on a 10 000-image subset and test on the full 10 000-image test set. Ten classes need two changes compared with Part B: the last layer has 10 neurons with `softmax` activation (one probability per digit), and the loss is `sparse_categorical_crossentropy`, which takes the integer labels directly.

In [ ]:
(x_train_full, y_train_full), (x_test_d, y_test_d) = keras.datasets.mnist.load_data()
print("full training set:", x_train_full.shape, y_train_full.shape, " test:", x_test_d.shape, y_test_d.shape,
      " pixel range:", x_train_full.min(), "-", x_train_full.max())

N_TRAIN = 10_000
x_train_d = x_train_full[:N_TRAIN].astype("float32")[..., None] / 255.0      # (10000, 28, 28, 1)
y_train_d = y_train_full[:N_TRAIN]
x_test_d  = x_test_d.astype("float32")[..., None] / 255.0

fig, ax = plt.subplots(2, 10, figsize=(10, 2.4))
for k, a in enumerate(ax.ravel()):
    a.imshow(x_train_d[k, :, :, 0], cmap="gray"); a.set_title(int(y_train_d[k]), fontsize=9); a.axis("off")
plt.tight_layout()

**Task D1.** Build and train a small CNN for MNIST. Suggested layout: `Input((28, 28, 1))`, two blocks of `Conv2D` (16 and 32 filters, 3×3, `activation='relu'`) each followed by `MaxPooling2D((2, 2))`, then `Flatten`, `Dense(64, activation='relu')`, `Dropout(0.3)` and `Dense(10, activation='softmax')`. Compile with `loss='sparse_categorical_crossentropy'`, optimizer `adam`, `metrics=['accuracy']`. Train for 4 epochs with `batch_size=64` and `validation_split=0.1`, then evaluate on the test set and print the test accuracy. Store the model in `mnist_model`, the next cell uses it.

In [ ]:
# Task D1: build, compile, train (4 epochs) and evaluate a CNN for MNIST. Keep the result in `mnist_model`.
keras.utils.set_random_seed(0)
mnist_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    # your code here
])
# your code here  (compile, fit, evaluate)

The next cell evaluates `mnist_model`: it prints the confusion matrix (rows: true digit, columns: predicted digit) and shows some of the digits the network gets wrong, with the prediction and the true label.

In [ ]:
from sklearn.metrics import confusion_matrix
from matplotlib.colors import LogNorm
prob_d = mnist_model.predict(x_test_d, verbose=0)
pred_d = prob_d.argmax(axis=1)
cm = confusion_matrix(y_test_d, pred_d)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap="Blues", norm=LogNorm())
for r in range(10):
    for c in range(10):
        ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=7, color="k" if r != c else "w")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("predicted digit"); ax.set_ylabel("true digit"); ax.set_title("confusion matrix (log colour scale)")
plt.tight_layout()

wrong = np.where(pred_d != y_test_d)[0]
fig, ax = plt.subplots(2, 8, figsize=(10, 3))
for a, i in zip(ax.ravel(), wrong[:16]):
    a.imshow(x_test_d[i, :, :, 0], cmap="gray"); a.axis("off")
    a.set_title(f"pred {pred_d[i]} (p={prob_d[i].max():.2f})\ntrue {y_test_d[i]}", fontsize=8)
plt.tight_layout()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[r, c], r, c) for r in range(10) for c in range(10) if r != c), reverse=True)[:6]
print("most frequent confusions (count, true -> predicted):", [(int(n), r, c) for n, r, c in pairs])

**Task D2.** From the confusion matrix and the list above: which digit pairs are confused most often, and why might that be? Write two or three sentences in the cell below. Then look at the misclassified examples: would you have read them correctly?

*Your answer here.*

## What to take to Lecture 2

- **Overfitting** shows up as a growing gap between training and validation (or test) curves: the training loss keeps falling while the validation loss turns upward. You saw it in Part A (accuracy panel) and in Part B (no-dropout model).
- **Parameter counts** matter. A dense layer on raw pixels needs $3072 \times 128$ weights and learns nothing about neighbouring pixels; a 3×3 convolution needs 28 weights per filter and sees the whole image.
- **Learning-rate decay** (the factor 0.999 in Part A) is the simplest learning-rate schedule. Lecture 2 covers momentum, Adam and schedules in more detail.
- **Early stopping** and **dropout** are regularisation methods: they limit how far the network can memorise the training set. Lecture 2 adds weight decay, data augmentation and batch normalisation.
- **Small data sets** call for cross-validation, an error bar, and transfer learning rather than a bigger network.